In [1]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
USE_COLS = [0, 1, 2, 3, 4, 5, 10, 12, 25]

In [3]:
dtype_map = {
    0: "string",   # Loan Sequence Number
    1: "string",   # Monthly Reporting Period (YYYYMM)
    2: "float64",  # Current Actual UPB
    3: "string",   # Current Loan Delinquency Status
    4: "Int64",    # Loan Age
    5: "Int64",    # Remaining Months to Legal Maturity
    10: "float64", # Current Interest Rate
    12: "string",  # DDLPI (YYYYMM)
    25: "float64"  # ELTV
}

In [19]:
quarter = "2013Q2"

In [20]:
input_file = f"performance_{quarter}.txt"

In [21]:
output_file = f"performance_{quarter}.parquet"

In [22]:
parquet_writer = None

In [23]:
for chunk in pd.read_csv(
    input_file,
    sep="|",
    header=None,
    usecols=USE_COLS,
    dtype=dtype_map,
    chunksize=100000,
    low_memory=False
):
    table = pa.Table.from_pandas(chunk, preserve_index=False)

    if parquet_writer is None:
        parquet_writer = pq.ParquetWriter(
            output_file,
            table.schema,
            compression="snappy"
        )

    parquet_writer.write_table(table)

if parquet_writer is not None:
    parquet_writer.close()